In [33]:
import pandas as pd
import re
import asyncio
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report

df = pd.read_csv("amore_final.csv")

print(df.head())
print(df.columns)

              상품명                                                URL  \
0    메이크온 시너지 마스크  https://www.amoremall.com/kr/ko/product/detail...   
1   가볍게 마시는 히알루론산  https://www.amoremall.com/kr/ko/product/detail...   
2  스킨 라이트 테라피 III  https://www.amoremall.com/kr/ko/product/detail...   
3   스킨 라이트 테라피 3S  https://www.amoremall.com/kr/ko/product/detail...   
4    젬 소노 테라피 릴리프  https://www.amoremall.com/kr/ko/product/detail...   

   price_original 용량_raw  용량_value 용량_unit  \
0          4000.0    20g      20.0       g   
1         35000.0    NaN       NaN     NaN   
2        350000.0    NaN       NaN     NaN   
3        473333.0    NaN       NaN     NaN   
4        325000.0    NaN       NaN     NaN   

                                                 전성분  
0  정제수, 부틸렌글라이콜, 글리세린, 나이아신아마이드, 1,2-헥산다이올, 판테놀, ...  
1                                                NaN  
2                                                NaN  
3                                                NaN  
4              

In [34]:
import sys
print(sys.executable)

/usr/local/bin/python3.12


#### 제품 Embedding과 추천 로직의 정확도를 높이기 위해 규칙을 적용하여 sub-category를 구성

In [35]:

import re
import pandas as pd

df = pd.read_csv("amore_final.csv")

def categorize(name):
    if pd.isna(name):
        return pd.Series({"category_rule": None, "subcategory_rule": None})

    text = name.lower().strip()

    if re.search(r"(시트|마스크|팩|딥시트|sheet)", text):
        return pd.Series({"category_rule": "스킨케어", "subcategory_rule": "마스크팩"})

    if "spf" in text or "선크림" in text or "선 크림" in text or "sun" in text:
        return pd.Series({"category_rule": "선케어", "subcategory_rule": "선크림"})

    if "선스틱" in text or "선 스틱" in text:
        return pd.Series({"category_rule": "선케어", "subcategory_rule": "선스틱"})

    if "선쿠션" in text or "선 쿠션" in text:
        return pd.Series({"category_rule": "선케어", "subcategory_rule": "선쿠션"})

    if re.search(r"(마스카라|픽서|lash)", text):
        return pd.Series({"category_rule": "메이크업", "subcategory_rule": "마스카라"})

    if re.search(r"(브로우|아이브로우|brow)", text):
        return pd.Series({"category_rule": "메이크업", "subcategory_rule": "아이브로우"})

    if re.search(r"(아이라이너|라이너|liner|펜슬)", text):
        return pd.Series({"category_rule": "메이크업", "subcategory_rule": "아이라이너"})

    if re.search(r"(섀도우|쉐도우|shadow|아이즈|eyez)", text):
        return pd.Series({"category_rule": "메이크업", "subcategory_rule": "아이섀도우"})

    if re.search(r"(틴트|립스틱|립 케어|립케어|lip)", text):
        
        if "틴트" in text:
            return pd.Series({"category_rule": "메이크업", "subcategory_rule": "립틴트"})
        if "스틱" in text:
            return pd.Series({"category_rule": "메이크업", "subcategory_rule": "립스틱"})
        return pd.Series({"category_rule": "메이크업", "subcategory_rule": "립제품"})

    if re.search(r"(앰플)", text):
        return pd.Series({"category_rule": "스킨케어", "subcategory_rule": "앰플"})

    if re.search(r"(세럼|serum)", text):
        return pd.Series({"category_rule": "스킨케어", "subcategory_rule": "세럼"})

    if re.search(r"(에센스|essence)", text):
        return pd.Series({"category_rule": "스킨케어", "subcategory_rule": "에센스"})

    if re.search(r"(크림|밤 )", text):
        return pd.Series({"category_rule": "스킨케어", "subcategory_rule": "크림"})

    if re.search(r"(리무버|remover)", text):
        return pd.Series({"category_rule": "클렌징", "subcategory_rule": "리무버"})

    if re.search(r"(바디|body)", text):
        return pd.Series({"category_rule": "바디", "subcategory_rule": "바디케어"})

    if re.search(r"(테라피|makeon|device|led)", text):
        return pd.Series({"category_rule": "디바이스", "subcategory_rule": "스킨기기"})

    if re.search(r"(히알루론산|g|mg|정|캡슐)", text):
        if "ml" not in text:  
            return pd.Series({"category_rule": "건기식", "subcategory_rule": "기능성식품"})

    return pd.Series({"category_rule": "기타", "subcategory_rule": "미분류"})

# 규칙 적용
df[["category_rule", "subcategory_rule"]] = df["상품명"].apply(categorize)

df[["상품명", "category_rule", "subcategory_rule"]].head(30)

,상품명,category_rule,subcategory_rule
0,메이크온 시너지 마스크,스킨케어,마스크팩
1,가볍게 마시는 히알루론산,건기식,기능성식품
2,스킨 라이트 테라피 III,디바이스,스킨기기
3,스킨 라이트 테라피 3S,디바이스,스킨기기
4,젬 소노 테라피 릴리프,디바이스,스킨기기
5,히알루로닉 스파 캡슐 24개입,건기식,기능성식품
6,마그네타이트 바디롤러,바디,바디케어
7,NaN,None,None
8,페이셜 부스팅 스파,기타,미분류
9,수분가득 콜라겐 시트 마스크 25ml,스킨케어,마스크팩


In [36]:
print("규칙으로 분류된 비율:", df["category_rule"].notna().mean())

print("\n=== category 분포 ===")
print(df["category_rule"].value_counts(dropna=False))

print("\n=== subcategory 분포 ===")
print(df["subcategory_rule"].value_counts(dropna=False))

규칙으로 분류된 비율: 0.9967213114754099

=== category 분포 ===
category_rule
기타      668
스킨케어    458
건기식     123
선케어     123
메이크업     91
바디       46
클렌징       8
None      5
디바이스      3
Name: count, dtype: int64

=== subcategory 분포 ===
subcategory_rule
미분류      668
크림       189
기능성식품    123
선크림      123
마스크팩      96
세럼        84
에센스       56
바디케어      46
앰플        33
마스카라      23
아이섀도우     18
립틴트       17
아이라이너     14
아이브로우     14
리무버        8
None       5
립스틱        4
스킨기기       3
립제품        1
Name: count, dtype: int64


#### ML 학습용 텍스트

In [37]:
# NaN 
df["상품명"] = df["상품명"].fillna("")
df["전성분"] = df["전성분"].fillna("")

# ML 입력 텍스트
df["ml_text"] = df["상품명"] + " " + df["전성분"]

mask_labeled = df["category_rule"].notna()

train_texts = df.loc[mask_labeled, "ml_text"]
y_cat = df.loc[mask_labeled, "category_rule"]
y_sub = df.loc[mask_labeled, "subcategory_rule"]

print("학습 데이터 수:", len(train_texts))
print(train_texts.head())

학습 데이터 수: 1520
0    메이크온 시너지 마스크 정제수, 부틸렌글라이콜, 글리세린, 나이아신아마이드, 1,2...
1                                       가볍게 마시는 히알루론산 
2                                      스킨 라이트 테라피 III 
3                                       스킨 라이트 테라피 3S 
4                                        젬 소노 테라피 릴리프 
Name: ml_text, dtype: object


#### 카테고리 ML 모델 학습

In [39]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report

# train / validation split
X_train_c, X_val_c, y_train_c, y_val_c = train_test_split(
    train_texts, y_cat,
    test_size=0.2,
    random_state=42,
    stratify=y_cat
)

# Pipeline 구성
cat_clf = Pipeline([
    ("tfidf", TfidfVectorizer(max_features=30000)),
    ("clf", LogisticRegression(
        max_iter=300,
        random_state=42
    ))
])

cat_clf.fit(X_train_c, y_train_c)

y_pred_c = cat_clf.predict(X_val_c)

print("=== Category classification report ===")
print(classification_report(y_val_c, y_pred_c))

=== Category classification report ===
              precision    recall  f1-score   support

         건기식       0.80      0.17      0.28        24
          기타       0.71      0.91      0.79       134
        디바이스       0.00      0.00      0.00         1
        메이크업       0.90      0.50      0.64        18
          바디       1.00      0.11      0.20         9
         선케어       0.96      0.92      0.94        24
        스킨케어       0.87      0.87      0.87        92
         클렌징       0.00      0.00      0.00         2

    accuracy                           0.78       304
   macro avg       0.65      0.43      0.46       304
weighted avg       0.80      0.78      0.75       304



/Users/mac/Library/Python/3.12/lib/python/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/mac/Library/Python/3.12/lib/python/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/mac/Library/Python/3.12/lib/python/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result

In [43]:
# 값이 2개 이상인 category만 사용
valid_sub = y_sub.value_counts()[y_sub.value_counts() >= 2].index

mask_valid = y_sub.isin(valid_sub)

train_texts_s = train_texts[mask_valid]
y_sub_s = y_sub[mask_valid]

print("유효 subcategory 개수:", len(valid_sub))


X_train_s, X_val_s, y_train_s, y_val_s = train_test_split(
    train_texts_s,
    y_sub_s,
    test_size=0.2,
    random_state=42,
    stratify=y_sub_s
)


sub_clf = Pipeline([
    ("tfidf", TfidfVectorizer(max_features=30000)),
    ("clf", LogisticRegression(
        max_iter=300,
        random_state=42
    ))
])

sub_clf.fit(X_train_s, y_train_s)

y_pred_s = sub_clf.predict(X_val_s)

print("=== Subcategory classification report ===")
print(classification_report(y_val_s, y_pred_s))

유효 subcategory 개수: 17
=== Subcategory classification report ===
              precision    recall  f1-score   support

       기능성식품       0.56      0.20      0.29        25
         리무버       0.00      0.00      0.00         1
         립스틱       0.00      0.00      0.00         1
         립틴트       0.00      0.00      0.00         3
        마스카라       1.00      0.20      0.33         5
        마스크팩       1.00      0.37      0.54        19
         미분류       0.60      0.96      0.74       134
        바디케어       0.00      0.00      0.00         9
         선크림       0.73      0.88      0.80        25
          세럼       1.00      0.29      0.45        17
       아이라이너       0.00      0.00      0.00         3
       아이브로우       0.00      0.00      0.00         3
       아이섀도우       0.00      0.00      0.00         3
          앰플       1.00      0.29      0.44         7
         에센스       0.50      0.09      0.15        11
          크림       0.74      0.66      0.69        38

    accuracy    

/Users/mac/Library/Python/3.12/lib/python/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/mac/Library/Python/3.12/lib/python/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/mac/Library/Python/3.12/lib/python/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result

#### 규칙 실패만 ML로 보정

In [44]:
mask_unlabeled = df["category_rule"].isna()

X_unlabeled = df.loc[mask_unlabeled, "ml_text"]

if len(X_unlabeled) > 0:

    cat_pred_unlabeled = cat_clf.predict(X_unlabeled)
    sub_pred_unlabeled = sub_clf.predict(X_unlabeled)

    df.loc[mask_unlabeled, "category_rule"] = cat_pred_unlabeled
    df.loc[mask_unlabeled, "subcategory_rule"] = sub_pred_unlabeled


df["category"] = df["category_rule"]
df["subcategory"] = df["subcategory_rule"]

df[["상품명", "category", "subcategory"]].head(20)

,상품명,category,subcategory
0,메이크온 시너지 마스크,스킨케어,마스크팩
1,가볍게 마시는 히알루론산,건기식,기능성식품
2,스킨 라이트 테라피 III,디바이스,스킨기기
3,스킨 라이트 테라피 3S,디바이스,스킨기기
4,젬 소노 테라피 릴리프,디바이스,스킨기기
5,히알루로닉 스파 캡슐 24개입,건기식,기능성식품
6,마그네타이트 바디롤러,바디,바디케어
7,,기타,미분류
8,페이셜 부스팅 스파,기타,미분류
9,수분가득 콜라겐 시트 마스크 25ml,스킨케어,마스크팩


In [46]:
final_cols = [
    "상품명",
    "URL",
    "brand",
    "product_id",
    "price_original",
    "용량_raw",
    "용량_value",
    "용량_unit",
    "전성분",
    "category",
    "subcategory"
]

final_cols_existing = [c for c in final_cols if c in df.columns]
final_df = df[final_cols_existing]

final_df.to_csv("amore_with_category.csv", index=False, encoding="utf-8-sig")

print("저장 완료 → amore_with_category.csv")
final_df.head()

저장 완료 → amore_with_category.csv


,상품명,URL,price_original,용량_raw,용량_value,용량_unit,전성분,category,subcategory
0,메이크온 시너지 마스크,https://www.amoremall.com/kr/ko/product/detail...,4000.0,20g,20.0,g,"정제수, 부틸렌글라이콜, 글리세린, 나이아신아마이드, 1,2-헥산다이올, 판테놀, ...",스킨케어,마스크팩
1,가볍게 마시는 히알루론산,https://www.amoremall.com/kr/ko/product/detail...,35000.0,NaN,NaN,NaN,,건기식,기능성식품
2,스킨 라이트 테라피 III,https://www.amoremall.com/kr/ko/product/detail...,350000.0,NaN,NaN,NaN,,디바이스,스킨기기
3,스킨 라이트 테라피 3S,https://www.amoremall.com/kr/ko/product/detail...,473333.0,NaN,NaN,NaN,,디바이스,스킨기기
4,젬 소노 테라피 릴리프,https://www.amoremall.com/kr/ko/product/detail...,325000.0,NaN,NaN,NaN,,디바이스,스킨기기
